# Week 7 · Multi-Agent Systems

**The problem.** A rider writes: "my e-bike battery died halfway, I pushed it to the dock, I want my money back". Resolving that means deciding what the rider is owed, issuing the right credit, opening a repair ticket, and replying. `POST /v1/resolve` does it today by sending every case to every specialist: slow, and specialists act on cases that are not theirs. By the end of this lab a triage agent picks the team, and the credits are exactly right.

That is several jobs with different rules and different permissions. Today you learn when to split one agent into a team, how to orchestrate the team with Microsoft Agent Framework, how to keep a person in the loop for actions that move money, and how to prove the split paid off.

| Part | What you do | Lecture notes |
|---|---|---|
| 1 | One agent with every tool: where it breaks, and what splitting costs | §1, §3 |
| 2 | Workflows from scratch: sequential, concurrent, and the other patterns | §5, §8 |
| 3 | Routing with a triage agent; approvals for actions | §6, §7, §10 |
| 4 | **Your turn:** measure the resolution team, add triage, measure again | §9, §11, §12 |
| 5 | Deploy and score | |

**You ship:** `POST /v1/resolve`, a triage agent plus specialists that act correctly. **You submit:** your URL.

**How to use this notebook.** Run the cells in order. Parts 1 to 3 show each pattern on the same support case; Part 4 is where you change the service; Part 5 ships it. The [lecture notes](lecture.md) explain the why behind each part.

**Optional, for depth:** [deep-dive.ipynb](deep-dive.ipynb) takes Agent Framework apart: workflow graphs and supersteps, switch-case routing, middleware, checkpoints that survive a restart, group chat and handoff, the cost of a team, and `evaluate_agent`. It is not graded and cannot change your score; open it after you submit.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())))
import course

course.setup()

## Part 1 · One agent with every tool

Start with the simplest design that could work: one agent that holds every tool the service has, from handbook search to issuing credit. Give it the battery case:

In [ ]:
from agent_framework import Agent

from app.config import get_settings
from app.state import AppState
from app.tools import get_station, issue_credit, open_ticket, quote_fare, recording, search_handbook

state = AppState(get_settings())
client = state.models.agent_client()

generalist = Agent(
    client=client,
    name="generalist",
    instructions="You resolve Lakeshore Ride support cases. Use the tools as needed, then reply to the rider.",
    tools=[search_handbook, quote_fare, get_station, issue_credit, open_ticket],
)

CASE = ("Rider R-1042 is a member. They took a 40-minute e-bike ride from LS-14; the battery died halfway "
        "and they pushed the bike to the dock. They want the e-bike charge back, and the bike should be checked.")

with recording(state) as calls:
    result = await generalist.run(CASE)

print(result.text, "\n")
for call in calls:
    print(f"{call['name']}({call['arguments']}) -> {call['result'] if call['name'] != 'search_handbook' else '...'}")

Check it against the handbook: the e-bike charge for a member is 40 × $0.18 = **$7.20**, refunded as ride credit, and the bike needs a `bike_repair` ticket at LS-14. Run the cell two or three times. A single agent often gets this case right; the trouble shows up as the prompt and the tool list grow:

- **Confused responsibilities:** one prompt holds billing rules, operations rules and tone rules. Each new rule can break another.
- **Too much power:** every part of the agent can issue credit, including the part answering a policy question.
- **Hard to debug:** when a credit is wrong, which "part" of the prompt caused it?

Splitting into specialists addresses all three, at a cost you should estimate before you build:

- **Calls and tokens.** Every agent is at least one model call with its own instructions. Anthropic reports that its multi-agent research system used about fifteen times the tokens of a chat ([How we built our multi-agent research system](https://www.anthropic.com/engineering/multi-agent-research-system)).
- **Errors compound.** If three agents each act correctly 95% of the time and fail independently, the chain is right about 86% of the time (0.95³).
- **Latency adds up** for every step that waits on another.

That trade-off is the whole topic of today.

## Part 2 · Workflows, from scratch

Before agents decide anything, learn the basic shapes. In a **workflow** *you* fix the order; in an **agent** the model decides. Most production systems are workflows with agents inside them.

| Pattern | Shape | Use it when | Agent Framework |
|---|---|---|---|
| Sequential | A → B → C | Each step refines the previous one (draft, then edit) | `SequentialBuilder` |
| Concurrent | A, B, C in parallel → gather | Independent views or sub-tasks | `ConcurrentBuilder` |
| Agent as tool | A manager calls specialists like functions and keeps control of the reply | One voice to the user, specialist work behind it | `agent.as_tool()` |
| Handoff | A passes control to B, who now owns the conversation | One owner at a time, chosen as you go | `HandoffBuilder` |
| Group chat | Agents take turns in a shared conversation | Debate, critique, consensus | `GroupChatBuilder` |
| Manager-led (Magentic) | A manager plans and assigns sub-tasks | Open-ended tasks with many steps | `MagenticBuilder` |

The difference between the third and fourth rows is who answers the user: with agents as tools the manager does; after a handoff the specialist does. OpenAI's [orchestration guide](https://developers.openai.com/api/docs/guides/agents/orchestration) draws the same line. The [deep dive](deep-dive.ipynb) runs group chat, handoff and agents as tools on this same case.

A small helper to print what each agent said:

In [ ]:
from agent_framework.orchestrations import ConcurrentBuilder, SequentialBuilder


def show(outputs) -> None:
    for output in outputs:
        items = output if isinstance(output, list) else (getattr(output, "messages", None) or [output])
        for message in items:
            text = getattr(message, "text", message)
            if text:
                print(f"[{getattr(message, 'author_name', None) or 'agent'}] {text}\n")

### Sequential: draft, then edit

In [ ]:
drafter = Agent(client=client, name="drafter",
                instructions="Draft a reply to the rider in three sentences. Acknowledge the problem and say what happens next.")
editor = Agent(client=client, name="editor",
               instructions="Edit the draft you receive: shorter, warmer, no promises the company cannot keep. Return only the final text.")

sequential = SequentialBuilder(participants=[drafter, editor]).build()
show((await sequential.run(CASE)).get_outputs())

### Concurrent: independent views at the same time

Two agents read the same case with different goals. They run in parallel, so total time is roughly the slower of the two, not the sum.

In [ ]:
import time

money = Agent(client=client, name="money", instructions="List only what the rider may be owed, with amounts. One line each.", tools=[search_handbook, quote_fare])
fleet = Agent(client=client, name="fleet", instructions="List only what the operations team must do. One line each.", tools=[get_station])

with recording(state):
    start = time.perf_counter()
    show((await ConcurrentBuilder(participants=[money, fleet]).build().run(CASE)).get_outputs())
print(f"{time.perf_counter() - start:.1f} s for both")

Notice the tools: `money` can compute fares but cannot open tickets; `fleet` can read stations but cannot touch money. That is **least privilege** for agents, the same principle you applied to the managed identity in Setup.

## Part 3 · Routing with a triage agent

Not every case needs every specialist. A **triage** agent reads the case and returns, as structured output, which specialists to involve. Routing becomes data your code can log, test and enforce.

In [ ]:
from typing import Literal

from pydantic import BaseModel


class TriageDecision(BaseModel):
    specialists: list[Literal["policy", "billing", "operations"]]
    reason: str


TRIAGE = """You are the triage lead for Lakeshore Ride Support. Decide which specialists must work on the case:
- policy: explains rules and policies from the handbook.
- billing: checks what a ride should have cost and issues ride credit when the rider is owed money.
- operations: checks stations and opens maintenance tickets for broken bikes, docks or stations.
Choose every specialist the case needs and no others."""

triage = Agent(client=client, name="triage", instructions=TRIAGE)
for case in [CASE, "A parent asks whether their 15-year-old can ride with them this weekend.",
             "Is LS-05 a good place to pick up an e-bike right now?"]:
    decision = (await triage.run(case, options={"response_format": TriageDecision})).value
    print(f"{decision.specialists}  <- {case[:70]}")

### Approvals: a person in the loop

Some actions should not happen on a model's word alone. Agent Framework can mark a tool as needing approval. When the model calls it, the run **pauses** and returns the pending call instead of executing it; your code shows it to a person and resumes the run with the decision. Here a credit tool asks every time:

In [ ]:
import json

from agent_framework import Message, tool

from app import domain


@tool(name="issue_credit", approval_mode="always_require")
def issue_credit_with_approval(rider_id: str, amount: float, reason: str) -> dict:
    """Give a rider ride credit in US dollars. A person approves every credit before it is issued."""
    return domain.issue_credit(rider_id, amount, reason)


cashier = Agent(client=client, name="cashier", tools=[quote_fare, issue_credit_with_approval],
                instructions="Refund riders who are owed money: compute the charge with quote_fare, then issue credit.")
session = cashier.create_session()
result = await cashier.run(CASE, session=session)

decisions = []
for request in result.user_input_requests:
    call = request.function_call
    arguments = json.loads(call.arguments) if isinstance(call.arguments, str) else call.arguments
    approved = arguments["amount"] <= 25  # stands in for a person pressing "approve"
    print(f"paused: {call.name}({arguments}) -> {'approved' if approved else 'rejected'}")
    decisions.append(Message(role="user", contents=[request.to_function_approval_response(approved)]))

if decisions:
    final = await cashier.run(decisions, session=session)
    print("\n" + final.text)
else:
    print(result.text, "\n(no credit was requested this time: run the cell again)")

The run stopped before any money moved, and the decision came from code you control. The approval is bound to the session, so a forged or replayed approval does not count. In the HTTP service nobody waits on the request, so `issue_credit` applies the same idea asynchronously: credits above $25 come back `escalated`, for a supervisor to review. Choose the gate by the cost of a mistake: approvals for irreversible or expensive actions, none for reads. See [tool approvals in Agent Framework](https://learn.microsoft.com/en-us/agent-framework/agents/tools/tool-approval).

## Part 4 · The resolution service

Read the starting version. Every case goes to all three specialists in parallel, then a writer drafts the reply:

In [ ]:
import inspect

from app.services import resolve

print(inspect.getsource(resolve.resolve))

Measure it. The harness sends the public cases and checks the route, every credit (rider, amount, status) and every ticket (category, station). A credit issued when none was owed scores zero for that case: in a real company that is money out of the door.

A multi-agent run makes many model calls per case. Your deployment has a budget of tokens **and** requests per minute (Week 5, Part 1: 100,000 tokens and 100 requests); when either is spent, Foundry answers 429, your service answers 503 with `Retry-After`, and the harness waits and retries. If a report shows many 503s, wait a minute before the next run. Fewer, smaller calls per case is also the cheapest way to go faster.

In [ ]:
baseline = course.check(7, in_process=True)

### Your turn

Open `app/services/resolve.py`:

1. **Add triage.** Use `TriageDecision` and the `TRIAGE` prompt from Part 3. Run the triage agent first, then build the team from `decision.specialists` only.
2. **Handle a team of one.** `ConcurrentBuilder` needs at least two participants. With one specialist, call `await team[0].run(request.case)` directly.
3. **Report the real route:** `route=["triage", *chosen, "writer"]`.
4. **Make the money right.** Read the actions in the proof: if a credit is wrong, the billing instructions are missing a rule. The rules are spread over the handbook: [`refunds-and-credits.md`](../app/data/handbook/refunds-and-credits.md) says who may approve what, and the pages on e-bikes, weather pauses, stations, and damage and lost bikes say what is owed. Write them as a **playbook**: one line per situation (a dead battery, a weather pause, a wrong fare, a stolen bike, and so on), each saying what is owed and which tool computes the amount. Typical gaps:
   - credit per-minute charges only, never the unlock fee;
   - when a fee was wrong or reduced by policy, credit the difference: a credit is the only way to fix a charge;
   - call `issue_credit` even above $25 (the system escalates it for review), and call it yourself: never ask the rider for permission. A reply that says "a credit is being processed" without the call gives the rider nothing;
   - no policy, no credit: goodwill credit is not a policy.

   If an amount is still wrong after you state the rule, stop prompting and compute it in code: give billing a small tool that returns the per-minute charges without the unlock fee (Week 6: models are unreliable calculators). If billing works out a credit but sometimes forgets to issue it, give it one tool that does both, for example one that compares a charge with the correct fare and credits the difference.
5. **Route by need, and report what really happened.** A request for money needs billing, even when nothing is owed. A broken bike or dock, or a request to check one, needs operations, which opens the ticket at the station where the problem is (and picks the right kind: a repair for something broken, rebalancing for a station that keeps running full or empty). The writer states every amount that matters (the corrected fee, each credit) and reports only actions the tools returned, with their status: an escalated credit waits for a supervisor; it was not issued.

Do steps 1 to 3 first and measure: routing alone usually moves the score most. Then work on the money. Deploy as soon as you beat the baseline; you can redeploy every improvement after that.

**100 is reachable.** Every public case has one right route and one right set of actions, and the detail lines name each case that missed and what the service did. Fix one cause at a time and re-run: one case that flips between runs means a rule is still ambiguous. Keep going until the local harness shows 100, then redeploy.

In [ ]:
improved = course.check(7, in_process=True)
print(f"\nbaseline {baseline['score']}  ->  now {improved['score']}")

Compare the two proofs case by case, not just the totals. Did triage reduce latency? Did it remove unowed actions? Did any case get worse because triage skipped a specialist it needed? That is the evidence for "the split paid off".

In [ ]:
def per_case(proof: dict) -> dict:
    """{case id: (seconds, route)} from the exchanges the harness recorded."""
    check = next(c for c in proof["checks"] if c["id"] == "w07.resolve.schema")
    cases = {}
    for exchange in check["exchanges"]:
        request = exchange["request"] if isinstance(exchange["request"], dict) else {}
        response = exchange["response"] if isinstance(exchange["response"], dict) else {}
        if "id" in request:
            cases[request["id"]] = (exchange["latency_ms"] / 1000, response.get("route"))
    return cases


before, after = per_case(baseline), per_case(improved)
for case_id in sorted(before):
    seconds_after, route_after = after.get(case_id, (0.0, None))
    print(f"{case_id}  {before[case_id][0]:5.1f} s -> {seconds_after:5.1f} s   route now {route_after}")

## Part 5 · Deploy and score

In [ ]:
url = course.deploy()
proof = course.check(7, url)

Submit your URL by the deadline in the [Week 7 README](README.md).

### Go further

- Try `HandoffBuilder` instead of the fixed fan-out: triage hands the case to one specialist, who may hand it on.
- Rebuild the team as **agents as tools**: one resolver agent that calls `billing.as_tool()` and `operations.as_tool()`. Compare route accuracy, latency and tokens.
- Add a **verifier** agent that checks each proposed credit against the policy before the writer runs.
- Count tokens per case: every `AgentResponse` has `usage_details`. Which specialist costs the most?
- **Deep dive (optional):** [deep-dive.ipynb](deep-dive.ipynb) builds workflows from executors and edges, routes with switch-case edges, enforces the $25 rule in middleware, pauses for a supervisor and resumes after a restart from a checkpoint, and measures what a team costs.

### What to remember

- Start with one agent. Split when responsibilities, permissions or failure modes diverge, and expect more tokens, more latency and compounding errors.
- Workflows fix the order; agents choose it. Production systems mix both.
- Routing with structured output makes the system's decisions visible and testable.
- Give each agent only the tools its job needs; put a person, or a rule in code, in front of actions that are hard to undo.
- Judge a multi-agent system by its **actions** and its **latency**, case by case.

**Learn more:** [Orchestrate a multi-agent solution using the Microsoft Agent Framework](https://learn.microsoft.com/en-us/training/modules/orchestrate-semantic-kernel-multi-agent-solution/) · [Building effective agents (Anthropic)](https://www.anthropic.com/engineering/building-effective-agents) · [How we built our multi-agent research system (Anthropic)](https://www.anthropic.com/engineering/multi-agent-research-system) · [Tool approvals in Agent Framework](https://learn.microsoft.com/en-us/agent-framework/agents/tools/tool-approval)